# Multi-Person Pose Estimation
## Eat The Beat BeatBusterz HipHop Groups 2nd Place — Dance Masters 2017

**Video:** Real-world competition footage (not studio, not controlled environment)  
**Challenge:** Stage lighting, camera movement, multiple dancers, costumes  
**Models:** YOLO26s-pose, RTMPose-m, ViTPose-base  

**This is domain shift evaluation:**  
Models trained on COCO (everyday scenes) tested on real dance competition footage.

In [ ]:
import torch
print('GPU:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))
device = 'cuda' if torch.cuda.is_available() else 'cpu'


In [ ]:
!pip install ultralytics --quiet
!pip install rtmlib --quiet
!pip install transformers --quiet
!pip install opencv-python-headless matplotlib pandas --quiet
 
import torch
!pip uninstall onnxruntime onnxruntime-gpu -y --quiet
!pip install onnxruntime-gpu==1.19.0 --quiet
 
import onnxruntime as ort
providers = ort.get_available_providers()
backend_device = 'cuda' if 'CUDAExecutionProvider' in providers else 'cpu'
print(f'ORT backend: {backend_device}')
print('Done')


In [ ]:
from google.colab import drive
drive.mount('/content/drive')
 
import os
import cv2
import numpy as np
import time
from PIL import Image
 
VIDEO_PATH  = '/content/drive/MyDrive/Files/Eat The Beat BeatBusterz HipHop Groups 2nd Place Dance Masters 2017.mp4'
RESULTS_DIR = '/content/drive/MyDrive/BeatBusterz_Results'
os.makedirs(RESULTS_DIR, exist_ok=True)
os.makedirs('/content/frames_bb', exist_ok=True)
 
cap = cv2.VideoCapture(VIDEO_PATH)
fps   = cap.get(cv2.CAP_PROP_FPS)
total = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
w     = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
h     = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
cap.release()
 
print(f'Video: {os.path.basename(VIDEO_PATH)}')
print(f'Resolution: {w}x{h}')
print(f'FPS: {fps:.1f}')
print(f'Total frames: {total}')
print(f'Duration: {total/fps:.1f}s = {total/fps/60:.1f}min')


In [ ]:
FRAME_STEP = max(1, int(fps // 5))
MAX_FRAMES = 100
 
start_frame = int(total * 0.10)
end_frame   = int(total * 0.90)
 
cap = cv2.VideoCapture(VIDEO_PATH)
cap.set(cv2.CAP_PROP_POS_FRAMES, start_frame)
 
frame_paths = []
frame_idx   = start_frame
saved       = 0
 
while cap.isOpened() and saved < MAX_FRAMES and frame_idx < end_frame:
    ret, frame = cap.read()
    if not ret:
        break
    if (frame_idx - start_frame) % FRAME_STEP == 0:
        path = f'/content/frames_bb/frame_{saved:04d}.jpg'
        cv2.imwrite(path, frame)
        frame_paths.append(path)
        saved += 1
    frame_idx += 1
 
cap.release()
print(f'Extracted {saved} frames')
print(f'Covering: {start_frame/fps:.1f}s — {end_frame/fps:.1f}s of video')


In [ ]:
from ultralytics import YOLO
from rtmlib import Body
from transformers import AutoModelForPoseEstimation, AutoProcessor
 
yolo_model = YOLO('yolo26s-pose.pt')
print('YOLO26s loaded')
 
rtm_model = Body(
    det='https://download.openmmlab.com/mmpose/v1/projects/rtmposev1/onnx_sdk/yolox_m_8xb8-300e_humanart-c2c7a14a.zip',
    det_input_size=(640, 640),
    pose='https://download.openmmlab.com/mmpose/v1/projects/rtmposev1/onnx_sdk/rtmpose-m_simcc-body7_pt-body7_420e-256x192-e48f03d0_20230504.zip',
    pose_input_size=(192, 256),
    to_openpose=False,
    backend='onnxruntime',
    device=backend_device
)
print(f'RTMPose-m loaded')
 
vit_processor = AutoProcessor.from_pretrained('usyd-community/vitpose-base-simple')
vit_model     = AutoModelForPoseEstimation.from_pretrained('usyd-community/vitpose-base-simple')
vit_model     = vit_model.to(device)
vit_model.eval()
print(f'ViTPose loaded on {device}')


In [ ]:
SKELETON_PAIRS = [
    (0,1),(0,2),(1,3),(2,4),
    (5,6),(5,7),(6,8),(7,9),(8,10),
    (5,11),(6,12),(11,12),
    (11,13),(12,14),(13,15),(14,16)
]
COLORS = ['#FF0000','#00FF00','#0000FF','#FF00FF','#FFFF00',
          '#00FFFF','#FF8000','#8000FF','#FF0080','#80FF00']
 
yolo_stats, rtm_stats, vit_stats = [], [], []
 
print(f'Running inference on {len(frame_paths)} frames...')
 
for i, path in enumerate(frame_paths):
    img = cv2.imread(path)
 
    
    t0  = time.time()
    res = yolo_model(path, verbose=False)
    t_y = time.time() - t0
    n_y = len(res[0].boxes) if res[0].boxes is not None else 0
    c_y = float(res[0].boxes.conf.mean().cpu()) if n_y > 0 else 0.0
    yolo_stats.append({'n': n_y, 'conf': c_y, 't': t_y})
 
    
    t0      = time.time()
    kp, sc  = rtm_model(img)
    t_r     = time.time() - t0
    n_r     = len(kp) if kp is not None else 0
    c_r     = float(np.mean(sc)) if n_r > 0 else 0.0
    rtm_stats.append({'n': n_r, 'conf': c_r, 't': t_r})
 
    
    t0    = time.time()
    n_v   = 0
    c_v   = 0.0
    if n_y > 0:
        boxes_xyxy = res[0].boxes.xyxy.cpu().numpy()
        img_pil    = Image.open(path).convert('RGB')
        all_confs  = []
        for box in boxes_xyxy:
            x1,y1,x2,y2 = box
            bi = [[[float(x1),float(y1),float(x2-x1),float(y2-y1)]]]
            inp = vit_processor(img_pil, boxes=bi, return_tensors='pt')
            inp = {k: v.to(device) for k,v in inp.items()}
            with torch.no_grad():
                out = vit_model(**inp)
            r = vit_processor.post_process_pose_estimation(out, bi)[0][0]
            all_confs.append(float(r['scores'].mean().cpu()))
            n_v += 1
        c_v = float(np.mean(all_confs))
    t_v = time.time() - t0
    vit_stats.append({'n': n_v, 'conf': c_v, 't': t_v})
 
    if (i+1) % 20 == 0:
        print(f'  {i+1}/{len(frame_paths)} | YOLO:{n_y}det | RTM:{n_r}det | ViT:{n_v}det')
 
print('\nDone!')
print(f'YOLO:    avg={np.mean([s["n"] for s in yolo_stats]):.1f} det | FPS={1/np.mean([s["t"] for s in yolo_stats]):.1f}')
print(f'RTMPose: avg={np.mean([s["n"] for s in rtm_stats]):.1f} det | FPS={1/np.mean([s["t"] for s in rtm_stats]):.1f}')
print(f'ViTPose: avg={np.mean([s["n"] for s in vit_stats]):.1f} det | FPS={1/np.mean([s["t"] for s in vit_stats]):.1f}')


In [ ]:
import matplotlib.pyplot as plt
 
start = int(len(frame_paths) * 0.2)
step  = max(1, (len(frame_paths) - start) // 10)
selected = frame_paths[start::step][:10]
 
fig, axes = plt.subplots(10, 3, figsize=(18, 45))
fig.suptitle('Eat The Beat BeatBusterz — Multi-Person Pose Estimation\n'
             'HipHop Groups 2nd Place — Dance Masters 2017',
             fontsize=14, y=1.005)
 
col_titles = ['YOLO26s-pose\n(Single-stage)',
              'RTMPose-m\n(Top-down CNN)',
              'ViTPose-base\n(Top-down Transformer)']
 
for row, frame_path in enumerate(selected):
    img_bgr = cv2.imread(frame_path)
    img_rgb = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB)
 
    
    axes[row,0].imshow(img_rgb)
    res = yolo_model(frame_path, verbose=False)
    n_y = 0
    if res[0].keypoints is not None:
        for p_idx in range(len(res[0].keypoints)):
            color = COLORS[p_idx % len(COLORS)]
            kpts  = res[0].keypoints[p_idx].xy[0].cpu().numpy()
            confs = res[0].keypoints[p_idx].conf[0].cpu().numpy()
            for i,j in SKELETON_PAIRS:
                if confs[i]>0.3 and confs[j]>0.3:
                    axes[row,0].plot([kpts[i][0],kpts[j][0]],[kpts[i][1],kpts[j][1]],
                                    color=color,linewidth=1.5)
            for k in range(17):
                if confs[k]>0.3:
                    axes[row,0].scatter(kpts[k][0],kpts[k][1],c=color,s=15,zorder=5)
            n_y += 1
    title = col_titles[0] if row==0 else ''
    axes[row,0].set_title(f'{title}', fontsize=9)
    axes[row,0].set_ylabel(f'Frame {start+row*step+1}\n{n_y} detected', fontsize=7)
    axes[row,0].axis('off')
 
    
    axes[row,1].imshow(img_rgb)
    kp_all, sc_all = rtm_model(img_bgr)
    n_r = len(kp_all) if kp_all is not None else 0
    for p_idx in range(n_r):
        color = COLORS[p_idx % len(COLORS)]
        kpts  = kp_all[p_idx]
        confs = sc_all[p_idx]
        for i,j in SKELETON_PAIRS:
            if confs[i]>0.3 and confs[j]>0.3:
                axes[row,1].plot([kpts[i][0],kpts[j][0]],[kpts[i][1],kpts[j][1]],
                                color=color,linewidth=1.5)
        for k in range(17):
            if confs[k]>0.3:
                axes[row,1].scatter(kpts[k][0],kpts[k][1],c=color,s=15,zorder=5)
    title = col_titles[1] if row==0 else ''
    axes[row,1].set_title(f'{title}', fontsize=9)
    axes[row,1].set_ylabel(f'{n_r} detected', fontsize=7)
    axes[row,1].axis('off')
 
    
    axes[row,2].imshow(img_rgb)
    n_v = 0
    if res[0].boxes is not None and len(res[0].boxes) > 0:
        boxes_xyxy = res[0].boxes.xyxy.cpu().numpy()
        img_pil    = Image.open(frame_path).convert('RGB')
        for p_idx, box in enumerate(boxes_xyxy):
            x1,y1,x2,y2 = box
            bi  = [[[float(x1),float(y1),float(x2-x1),float(y2-y1)]]]
            inp = vit_processor(img_pil, boxes=bi, return_tensors='pt')
            inp = {k: v.to(device) for k,v in inp.items()}
            with torch.no_grad():
                out = vit_model(**inp)
            r     = vit_processor.post_process_pose_estimation(out,bi)[0][0]
            kpts  = r['keypoints'].cpu().numpy()
            confs = r['scores'].cpu().numpy()
            color = COLORS[p_idx % len(COLORS)]
            for i,j in SKELETON_PAIRS:
                if confs[i]>0.3 and confs[j]>0.3:
                    axes[row,2].plot([kpts[i][0],kpts[j][0]],[kpts[i][1],kpts[j][1]],
                                    color=color,linewidth=1.5)
            for k in range(17):
                if confs[k]>0.3:
                    axes[row,2].scatter(kpts[k][0],kpts[k][1],c=color,s=15,zorder=5)
            n_v += 1
    title = col_titles[2] if row==0 else ''
    axes[row,2].set_title(f'{title}', fontsize=9)
    axes[row,2].set_ylabel(f'{n_v} detected', fontsize=7)
    axes[row,2].axis('off')
 
plt.tight_layout()
plt.savefig('beatbusterz_visual.png', dpi=100, bbox_inches='tight')
plt.show()
print('Visual saved')


In [ ]:
import pandas as pd
 
summary = pd.DataFrame([
    {
        'Model':       'YOLO26s-pose',
        'Architecture': 'Single-stage',
        'Avg Detections': round(np.mean([s['n'] for s in yolo_stats]), 2),
        'Avg Confidence': round(np.mean([s['conf'] for s in yolo_stats]), 3),
        'FPS':            round(1/np.mean([s['t'] for s in yolo_stats]), 1)
    },
    {
        'Model':       'RTMPose-m',
        'Architecture': 'Top-down CNN',
        'Avg Detections': round(np.mean([s['n'] for s in rtm_stats]), 2),
        'Avg Confidence': round(np.mean([s['conf'] for s in rtm_stats]), 3),
        'FPS':            round(1/np.mean([s['t'] for s in rtm_stats]), 1)
    },
    {
        'Model':       'ViTPose-base',
        'Architecture': 'Top-down Transformer',
        'Avg Detections': round(np.mean([s['n'] for s in vit_stats]), 2),
        'Avg Confidence': round(np.mean([s['conf'] for s in vit_stats]), 3),
        'FPS':            round(1/np.mean([s['t'] for s in vit_stats]), 1)
    }
])
 
print('=== RESULTS — BeatBusterz HipHop Competition ===')
print(summary.to_string(index=False))
 
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
colors  = ['#2196F3', '#4CAF50', '#FF9800']
models  = ['YOLO26s', 'RTMPose-m', 'ViTPose-b']
frames_x = range(len(frame_paths))
 
axes[0].plot(frames_x, [s['n'] for s in yolo_stats], color=colors[0], label='YOLO26s',   alpha=0.8)
axes[0].plot(frames_x, [s['n'] for s in rtm_stats],  color=colors[1], label='RTMPose-m', alpha=0.8)
axes[0].plot(frames_x, [s['n'] for s in vit_stats],  color=colors[2], label='ViTPose-b', alpha=0.8)
axes[0].set_xlabel('Frame')
axes[0].set_ylabel('Detections')
axes[0].set_title('Detections per Frame Over Time')
axes[0].legend()
axes[0].grid(alpha=0.3)
 
bars = axes[1].bar(models, summary['Avg Detections'], color=colors, alpha=0.9)
for bar, v in zip(bars, summary['Avg Detections']):
    axes[1].text(bar.get_x()+bar.get_width()/2, v+0.05,
                f'{v:.1f}', ha='center', fontweight='bold')
axes[1].set_ylabel('Avg Detections per Frame')
axes[1].set_title('Average Detections')
axes[1].grid(axis='y', alpha=0.3)
 
bars = axes[2].bar(models, summary['FPS'], color=colors, alpha=0.9)
for bar, v in zip(bars, summary['FPS']):
    axes[2].text(bar.get_x()+bar.get_width()/2, v+0.2,
                f'{v:.1f}', ha='center', fontweight='bold')
axes[2].set_ylabel('FPS')
axes[2].set_title('Inference Speed (GPU T4)')
axes[2].grid(axis='y', alpha=0.3)
 
plt.suptitle('BeatBusterz HipHop — Pose Estimation Analysis', fontsize=13)
plt.tight_layout()
plt.savefig('beatbusterz_metrics.png', dpi=150, bbox_inches='tight')
plt.show()
 
summary.to_csv('beatbusterz_summary.csv', index=False)


In [ ]:
import shutil
 
files = [
    'beatbusterz_visual.png',
    'beatbusterz_metrics.png',
    'beatbusterz_summary.csv'
]
for f in files:
    shutil.copy(f, f'{RESULTS_DIR}/{f}')
    print(f'Saved: {f}')
 
print(f'\nAll results → BeatBusterz_Results/')
print()
print('=== FINAL SUMMARY ===')
print(summary.to_string(index=False))
